# Kinyarwanda Multidomain Text Collection

## Practicum objective

Build a clean, punctuated Kinyarwanda text collection across five domains:

| Domain | Clean-token target |
|---|---:|
| Health | 200,000 |
| Finance | 200,000 |
| Agriculture | 200,000 |
| Telecoms | 200,000 |
| Law | 200,000 |
| **Total** | **1,000,000** |

For domains with enough source material, this notebook keeps approximately **300,000 candidate tokens** so quality filtering can remove records while still leaving at least 200,000 clean tokens.

For **Telecoms**, the current decision is to use **only the KINNEWS `technology` category**. The Rwanda Gazette remains a **Law-only** source.

## Data sources

| Domain | Source | Use | License / status |
|---|---|---|---|
| Health | [DigitalUmuganda/Afrivoice_Kinyarwanda](https://huggingface.co/datasets/DigitalUmuganda/Afrivoice_Kinyarwanda) | Health transcripts | CC BY 4.0 |
| Finance | [DigitalUmuganda/Afrivoice_Kinyarwanda](https://huggingface.co/datasets/DigitalUmuganda/Afrivoice_Kinyarwanda) | Financial Services transcripts | CC BY 4.0 |
| Agriculture | [DigitalUmuganda/Afrivoice_Kinyarwanda](https://huggingface.co/datasets/DigitalUmuganda/Afrivoice_Kinyarwanda) | Agriculture transcripts | CC BY 4.0 |
| H/F/A access manifest | [Professor/kinyarwanda-speech-data](https://huggingface.co/datasets/Professor/kinyarwanda-speech-data) | Lightweight manifest used to access text/domain labels without downloading audio | Underlying source recorded as Afrivoice |
| Law | [DigitalUmuganda/NMT_Rwandan-Gazette_parallel_data_en_kin](https://huggingface.co/datasets/DigitalUmuganda/NMT_Rwandan-Gazette_parallel_data_en_kin) | Kinyarwanda Official Gazette text | CC BY 4.0 |
| Telecoms | [KINNEWS official repository](https://github.com/Andrews2017/KINNEWS-and-KIRNEWS-Corpus) | Records whose original `en_label` is `technology`; article `content` is used as `raw_text` | MIT |

## Encoding and text-cleaning policy

Earlier versions of this notebook showed `y‚Äôurugaga`, `caf√©`, `¬´` and `10¬∞` in the Law and Telecom text and treated them as mojibake in the source data. **That was a display problem, not a data problem.** The exported CSVs were UTF-8 *without* a BOM, and Excel / Numbers on macOS open such files as MacRoman, which turns `’` into `‚Äô`, `«` into `¬´` and `°` into `¬∞`. The raw KINNEWS and Gazette text is already valid UTF-8.

This notebook therefore does three things:

1. **Exports every CSV as `utf-8-sig`** (UTF-8 with BOM) so spreadsheets open it correctly. The JSONL stays plain UTF-8.
2. **Keeps `repair_mojibake()` as a safety net** for genuinely corrupted source text. On the current sources it changes almost nothing, so its "rows changed" counts must not be read as encoding fixes.
3. **Applies one shared `clean_text()`** to Law and Telecom before deduplication and token counting. It removes what really is in the source: NBSP, `\r`/`\n`/`\t`, double spaces, spaces before punctuation, zero-width characters, emoji and Cyrillic. It also unifies typographic quotes/apostrophes (`’ ‘ ‟ “ ” « »`) to plain ASCII `'` and `"` so all five domains use the same apostrophe as the Afrivoice domains. Law rows that contain English text (parallel-corpus leakage) are dropped.

Separate audit CSVs are still produced for Law and KINNEWS.

### Shared JSONL schema

```json
{"id":"kin_health_000001","language":"kin","domain":"health","raw_text":"...","source_dataset":"dataset name","source_split":"train","license":"CC-BY-4.0","review_status":"unreviewed","notes":""}
```

`raw_text` is preserved after source-level encoding repair. Other linguistic normalization should be done later through an explicit cleaning step.

## 1. Setup

This notebook is designed for Kaggle with Internet enabled.

**Important:** `/kaggle/working` is temporary. After generating the outputs, use **Save Version** in Kaggle and/or download the ZIP created at the end.

In [3]:
!pip -q install datasets regex huggingface_hub gdown

import os
import glob
import shutil
import zipfile
import subprocess
import unicodedata
import pandas as pd
import regex as re
from pathlib import Path
from datasets import load_dataset
from huggingface_hub import hf_hub_download

pd.set_option("display.max_colwidth", 300)

## 2. Configuration and helper functions

The token counter is Unicode-aware and keeps apostrophe-connected forms together.

In [4]:
COLUMNS = [
    "id",
    "language",
    "domain",
    "raw_text",
    "source_dataset",
    "source_split",
    "license",
    "review_status",
    "notes",
]

DOMAINS = [
    "health",
    "finance",
    "agriculture",
    "telecoms",
    "law",
]

FINAL_TARGET = 200_000
CANDIDATE_TARGET = 300_000

WORD_PATTERN = re.compile(
    r"\p{L}+(?:['’]\p{L}+)*"
)

def count_words(text):
    if not isinstance(text, str):
        return 0
    return len(WORD_PATTERN.findall(text))

def take_until_token_target(
    df,
    target_tokens=CANDIDATE_TARGET,
    seed=42
):
    work = df.copy()

    work = work[
        work["raw_text"]
        .astype(str)
        .str.strip()
        .ne("")
    ].copy()

    work = work.drop_duplicates(
        subset=["raw_text"]
    )

    work["word_count"] = (
        work["raw_text"].apply(count_words)
    )

    work = work.sample(
        frac=1,
        random_state=seed
    ).reset_index(drop=True)

    work["cumulative_tokens"] = (
        work["word_count"].cumsum()
    )

    selected = work[
        work["cumulative_tokens"] <= target_tokens
    ].copy()

    if (
        selected["word_count"].sum() < target_tokens
        and len(selected) < len(work)
    ):
        selected = pd.concat(
            [
                selected,
                work.iloc[[len(selected)]]
            ],
            ignore_index=True
        )

    return selected.drop(
        columns=["cumulative_tokens"],
        errors="ignore"
    )


# ---------------------------------------------------------
# GLOBAL SOURCE-LEVEL ENCODING REPAIR
# Used for BOTH Rwanda Gazette/Law and KINNEWS/Telecom.
# ---------------------------------------------------------

def build_mojibake_replacements():
    """
    Build targeted replacements for Unicode characters whose UTF-8
    bytes were decoded using common single-byte encodings.
    """
    correct_chars = []

    # Latin-1 Supplement + Latin Extended-A
    correct_chars.extend(
        chr(cp) for cp in range(0x00A0, 0x0180)
    )

    # General Punctuation
    correct_chars.extend(
        chr(cp) for cp in range(0x2000, 0x2070)
    )

    # Euro symbol
    correct_chars.append("€")

    mapping = {}

    for correct in correct_chars:
        utf8_bytes = correct.encode("utf-8")

        for wrong_encoding in [
            "mac_roman",
            "cp1252",
            "latin1",
        ]:
            try:
                broken = utf8_bytes.decode(wrong_encoding)
            except UnicodeDecodeError:
                continue

            if broken != correct and len(broken) >= 2:
                mapping[broken] = correct

    # Replace longest malformed sequences first.
    return dict(
        sorted(
            mapping.items(),
            key=lambda item: len(item[0]),
            reverse=True,
        )
    )


MOJIBAKE_REPLACEMENTS = build_mojibake_replacements()


def repair_mojibake(text):
    """
    Repair known mojibake substrings while leaving surrounding
    Kinyarwanda text unchanged.
    """
    if not isinstance(text, str):
        return text

    repaired = text

    # Repeated passes can repair text corrupted more than once.
    for _ in range(4):
        before = repaired

        for broken, correct in MOJIBAKE_REPLACEMENTS.items():
            if broken in repaired:
                repaired = repaired.replace(broken, correct)

        if repaired == before:
            break

    return unicodedata.normalize("NFC", repaired)


# High-confidence markers seen in the corrupted source text.
SUSPICIOUS_MOJIBAKE = [
    "√",
    "‚Ä",
    "â€",
    "Ã",
    "Â",
    "¬´",
    "¬ª",
    "Äô",
    "Äú",
    "Äù",
]


def has_suspicious_mojibake(text):
    if not isinstance(text, str):
        return False

    return any(
        marker in text
        for marker in SUSPICIOUS_MOJIBAKE
    )


print(
    "Generated global mojibake replacement patterns:",
    len(MOJIBAKE_REPLACEMENTS)
)

# ---------------------------------------------------------
# SHARED TEXT CLEANING (Law + Telecom)
# The "special characters" seen in spreadsheets were a viewer
# decoding issue (UTF-8 read as MacRoman). This function cleans
# what is genuinely present in the source text.
# ---------------------------------------------------------

INVISIBLE = re.compile(
    r"[\u200B-\u200F\u202A-\u202E\u2060\uFEFF\u00AD]"
)

CHAR_MAP = {
    "\u2019": "'", "\u2018": "'", "\u201B": "'", "\u201F": "'",  # ’ ‘ ‛ ‟
    "\u201C": '"', "\u201D": '"', "\u201E": '"',                  # “ ” „
    "\u00AB": '"', "\u00BB": '"',                                  # « »
    "\u2013": "-", "\u2014": "-", "\u2026": "...",                # – — …
    "\u00BA": "°", "\u2044": "/",
}

NON_TEXT = re.compile(
    r"[^\p{L}\p{N}\p{P}\p{S}\p{Z}]|[\U0001F000-\U0001FFFF\u0400-\u04FF]"
)

ENGLISH = re.compile(
    r"\b(the|and|of|shall|article)\b",
    re.I
)


def clean_text(t):
    if not isinstance(t, str):
        return t

    t = unicodedata.normalize("NFC", t)
    t = INVISIBLE.sub("", t)
    t = "".join(CHAR_MAP.get(c, c) for c in t)

    # Whitespace first (\r \n \t NBSP), so words are never glued together.
    t = re.sub(r"\s+", " ", t)

    # Then emoji, Cyrillic and any remaining control characters.
    t = NON_TEXT.sub("", t)

    t = re.sub(r"\s+", " ", t).strip()
    t = re.sub(r"\s+([,.!?;:])", r"\1", t)
    return t


Generated global mojibake replacement patterns: 815


# Part A — Health, Finance, and Agriculture

## 3. Load the speech-text manifest

The lightweight manifest gives access to the text and domain metadata without downloading the full audio collection.

The source labels are mapped as follows:

- `health` → `health`
- `financial` → `finance`
- `agriculture` → `agriculture`

In [5]:
manifest_path = hf_hub_download(
    repo_id="Professor/kinyarwanda-speech-data",
    filename="manifest.parquet",
    repo_type="dataset",
)

speech_df = pd.read_parquet(
    manifest_path
)

print("Manifest rows:", len(speech_df))
print("Columns:", speech_df.columns.tolist())

print("\nDomain counts:")
print(
    speech_df["domain"]
    .value_counts(dropna=False)
)

manifest.parquet:   0%|          | 0.00/24.0M [00:00<?, ?B/s]

Manifest rows: 187151
Columns: ['key', 'shard', 'text', 'duration', 'dataset_id', 'source', 'split', 'speaker_id', 'gender', 'domain', 'snr', 'lufs', 'dbfs', 'clip_ratio', 'sil_ratio', 'has_disfluency']

Domain counts:
domain
agriculture    37651
financial      37286
government     36909
health         36660
education      34653
None            3992
Name: count, dtype: int64


In [6]:
speech_needed = speech_df[
    speech_df["domain"].isin(
        ["health", "financial", "agriculture"]
    )
].copy()

speech_needed["domain"] = (
    speech_needed["domain"].replace({
        "financial": "finance"
    })
)

hfa_rows = []

for domain in [
    "health",
    "finance",
    "agriculture"
]:

    temp = speech_needed[
        speech_needed["domain"] == domain
    ].copy()

    temp = temp[
        temp["text"].apply(
            lambda x:
            isinstance(x, str)
            and bool(x.strip())
        )
    ].copy()

    temp = temp.drop_duplicates(
        subset=["text"]
    )

    temp["word_count"] = (
        temp["text"].apply(count_words)
    )

    temp = temp.sample(
        frac=1,
        random_state=42
    ).reset_index(drop=True)

    temp["cumulative_tokens"] = (
        temp["word_count"].cumsum()
    )

    selected = temp[
        temp["cumulative_tokens"]
        <= CANDIDATE_TARGET
    ].copy()

    if (
        selected["word_count"].sum()
        < CANDIDATE_TARGET
        and len(selected) < len(temp)
    ):
        selected = pd.concat(
            [
                selected,
                temp.iloc[[len(selected)]]
            ],
            ignore_index=True
        )

    for i, (_, row) in enumerate(
        selected.iterrows(),
        start=1
    ):

        hfa_rows.append({
            "id":
                f"kin_{domain}_{i:06d}",
            "language":
                "kin",
            "domain":
                domain,
            "raw_text":
                str(row["text"]).strip(),
            "source_dataset":
                "DigitalUmuganda/Afrivoice_Kinyarwanda",
            "source_split":
                str(row.get("split", "unknown")),
            "license":
                "CC-BY-4.0",
            "review_status":
                "unreviewed",
            "notes":
                (
                    "Loaded through Professor/"
                    "kinyarwanda-speech-data manifest; "
                    "verify human-reference transcript "
                    "provenance before final acceptance."
                ),
        })

candidate_hfa = pd.DataFrame(
    hfa_rows
)

candidate_hfa["word_count"] = (
    candidate_hfa["raw_text"]
    .apply(count_words)
)

hfa_summary = (
    candidate_hfa
    .groupby("domain")
    .agg(
        records=("raw_text", "size"),
        tokens=("word_count", "sum"),
    )
)

display(hfa_summary)

,records,tokens
domain,,
agriculture,11520,300022
finance,10902,300015
health,10731,300015


In [7]:
candidate_hfa[COLUMNS].to_csv(
    "/kaggle/working/kin_hfa_candidates.csv",
    index=False,
    encoding="utf-8-sig"
)

print(
    "Saved: "
    "/kaggle/working/kin_hfa_candidates.csv"
)

Saved: /kaggle/working/kin_hfa_candidates.csv


# Part B — Law

## 4. Load, repair, and audit the Rwanda Official Gazette corpus

The Kinyarwanda side of the Gazette corpus is used **only for Law**.

Before any Law deduplication or token counting:

1. the original `kin` text is retained temporarily for comparison;
2. the global mojibake repair is applied;
3. encoding changes are recorded;
4. suspicious residual sequences are audited;
5. only then are exact duplicates removed and tokens counted.

No Gazette passage is reassigned to Telecom in this notebook.

In [8]:
gazette = load_dataset(
    "DigitalUmuganda/"
    "NMT_Rwandan-Gazette_parallel_data_en_kin"
)

for split_name, ds in gazette.items():
    print(
        split_name,
        ds.num_rows,
        ds.column_names
    )

law_rows = []

for split_name, ds in gazette.items():

    for row in ds:

        original_text = row.get("kin")

        if (
            not isinstance(original_text, str)
            or not original_text.strip()
        ):
            continue

        repaired_text = clean_text(
            repair_mojibake(original_text)
        )

        law_rows.append({
            "id":
                f"kin_law_raw_{len(law_rows)+1:06d}",
            "language":
                "kin",
            "domain":
                "law",
            "raw_text":
                repaired_text,
            "source_dataset":
                (
                    "DigitalUmuganda/"
                    "NMT_Rwandan-Gazette_"
                    "parallel_data_en_kin"
                ),
            "source_split":
                split_name,
            "license":
                "CC-BY-4.0",
            "review_status":
                "unreviewed",
            "notes":
                "Source-level mojibake repair and clean_text() applied before deduplication.",
            "_raw_text_before":
                original_text.strip(),
        })

law_df = pd.DataFrame(
    law_rows
)

law_df["encoding_changed"] = (
    law_df["raw_text"]
    != law_df["_raw_text_before"]
)

law_df["still_suspicious"] = (
    law_df["raw_text"]
    .apply(has_suspicious_mojibake)
)

print(
    "\nLaw rows changed by encoding repair:",
    int(law_df["encoding_changed"].sum())
)

print(
    "Law rows still containing suspicious mojibake:",
    int(law_df["still_suspicious"].sum())
)

law_encoding_audit = law_df[
    law_df["encoding_changed"]
    | law_df["still_suspicious"]
][
    [
        "source_split",
        "_raw_text_before",
        "raw_text",
        "encoding_changed",
        "still_suspicious",
    ]
].copy()

if len(law_encoding_audit) > 0:
    print(
        "\nLaw encoding audit sample:"
    )
    display(
        law_encoding_audit.head(20)
    )
else:
    print(
        "\nNo Law encoding changes or suspicious residuals detected."
    )

# Deduplicate only AFTER encoding repair.
law_df = (
    law_df
    .drop_duplicates(
        subset=["raw_text"]
    )
    .reset_index(drop=True)
)

# Drop Gazette rows that contain English text (parallel-corpus leakage).
law_df["has_english"] = (
    law_df["raw_text"]
    .apply(lambda t: bool(ENGLISH.search(t)))
)

print(
    "\nLaw rows containing English text (dropped):",
    int(law_df["has_english"].sum())
)

law_df = (
    law_df[~law_df["has_english"]]
    .drop(columns="has_english")
    .reset_index(drop=True)
)

law_df["word_count"] = (
    law_df["raw_text"]
    .apply(count_words)
)

print(
    "\nUnique repaired Gazette records:",
    len(law_df)
)

print(
    "Total repaired Gazette tokens:",
    int(law_df["word_count"].sum())
)

README.md:   0%|          | 0.00/955 [00:00<?, ?B/s]

gazette_train.tsv:   0%|          | 0.00/50.7M [00:00<?, ?B/s]

gazette_val.tsv: 0.00B [00:00, ?B/s]

gazette_test.tsv: 0.00B [00:00, ?B/s]

Generating train split:   0%|          | 0/165184 [00:00<?, ? examples/s]

Generating validation split:   0%|          | 0/2554 [00:00<?, ? examples/s]

Generating test split:   0%|          | 0/2555 [00:00<?, ? examples/s]

train 165184 ['kin', 'eng']
validation 2554 ['kin', 'eng']
test 2555 ['kin', 'eng']

Law rows changed by encoding repair: 142082
Law rows still containing suspicious mojibake: 0

Law encoding audit sample:


,source_split,_raw_text_before,raw_text,encoding_changed,still_suspicious
0,train,"(2) uruhushya rutanzwe rugomba, igihe cyose, kumanikwa ahantu hagaragara ku cyicaro gikuru cy’ikigo cy’imari iciriritse cyakira amafaranga abitswa kandi kopi zacyo na zo zikamanikwa kuri buri shami ryacyo.","(2) uruhushya rutanzwe rugomba, igihe cyose, kumanikwa ahantu hagaragara ku cyicaro gikuru cy'ikigo cy'imari iciriritse cyakira amafaranga abitswa kandi kopi zacyo na zo zikamanikwa kuri buri shami ryacyo.",True,False
1,train,"Mu rwego rwo gushyiraho imirongo ngenderwaho no gushyira mu bikorwa ingamba zo gucunga imyenda ya leta buri mwaka, minisitiri agisha inama banki nkuru y’u rwanda, mu rwego rwo kwirinda kunyuranya na politiki y’ingengo y’imari, politiki yo gucunga imyenda, politiki y’ubucuruzi na politiki y’ivunj...","Mu rwego rwo gushyiraho imirongo ngenderwaho no gushyira mu bikorwa ingamba zo gucunga imyenda ya leta buri mwaka, minisitiri agisha inama banki nkuru y'u rwanda, mu rwego rwo kwirinda kunyuranya na politiki y'ingengo y'imari, politiki yo gucunga imyenda, politiki y'ubucuruzi na politiki y'ivunj...",True,False
3,train,"Article 11: garanties admissibles hitawe ku bipimo ntarengwa ku myenda minini, banki ishobora, mu bushishozi bwayo, gutanga avansi cyangwa gutanga inguzanyo iri hejuru ya 25%, ariko itarengeje 50% by’imari shingiro bwite yayo, iyo igihe cyo kwishyura iyo nguzanyo kitarenga imyaka itanu (5) kandi...","Article 11: garanties admissibles hitawe ku bipimo ntarengwa ku myenda minini, banki ishobora, mu bushishozi bwayo, gutanga avansi cyangwa gutanga inguzanyo iri hejuru ya 25%, ariko itarengeje 50% by'imari shingiro bwite yayo, iyo igihe cyo kwishyura iyo nguzanyo kitarenga imyaka itanu (5) kandi...",True,False
4,train,"Ikigo cy’igihugu kigomba kubanza kubemeza mbere y’uko batangira imirimo yabo, mu gihe kitarenze iminsi irindwi (7) y’akazi ibarwa uhereye igihe cyaboneye raporo y’aya matora.","Ikigo cy'igihugu kigomba kubanza kubemeza mbere y'uko batangira imirimo yabo, mu gihe kitarenze iminsi irindwi (7) y'akazi ibarwa uhereye igihe cyaboneye raporo y'aya matora.",True,False
5,train,"« uwegukanye isoko yemerewe kwifashisha ibimenyetso byose kugira ngo agaragaze uruhare rw’urwego rutanga isoko asaba kwishyurwa indishyi iyo ariyo yose, cyangwa se yerekana impamvu atubahirije igikorwa iki n’iki cyari mu nshingano ze, cyangwa asaba gusubizwa igice cyangwa amafaranga yose yakores...",""" uwegukanye isoko yemerewe kwifashisha ibimenyetso byose kugira ngo agaragaze uruhare rw'urwego rutanga isoko asaba kwishyurwa indishyi iyo ariyo yose, cyangwa se yerekana impamvu atubahirije igikorwa iki n'iki cyari mu nshingano ze, cyangwa asaba gusubizwa igice cyangwa amafaranga yose yakores...",True,False
6,train,"Dusubiye ku iteka rya perezida n˚ 36/01 ryo ku wa 07/09/2012 rishyiraho imishahara n‟ibindi bigenerwa umuyobozi n‟umuyobozi wungirije ba banki nkuru y‟ u rwanda, cyane cyane mu ngingo zaryo iya 3 n‟iya 4;","Dusubiye ku iteka rya perezida n˚ 36/01 ryo ku wa 07/09/2012 rishyiraho imishahara n'ibindi bigenerwa umuyobozi n'umuyobozi wungirije ba banki nkuru y' u rwanda, cyane cyane mu ngingo zaryo iya 3 n'iya 4;",True,False
8,train,Article premier : objet du présent règlement aya mabwiriza rusange ashyiraho uburyo abatanga serivisi zo kwishyurana bemererwa.,Article premier: objet du présent règlement aya mabwiriza rusange ashyiraho uburyo abatanga serivisi zo kwishyurana bemererwa.,True,False
9,train,"By’umwihariko, umubitsi w’impapurompamo z’ubutaka afite ububasha cyane cyane muri ibi bikurikira:","By'umwihariko, umubitsi w'impapurompamo z'ubutaka afite ububasha cyane cyane muri ibi bikurikira:",True,False
10,train,5° andi makuru asobanura urwego rw’ubugenzuzi rwasanga ari ingenzi kugira ngo rugere ku kigamijwe mu gukora igenzura ryo ku rwego rw’itsinda.,5° andi makuru asobanura urwego rw'ubugenzuzi rwasanga ari ingenzi kugira ngo rugere ku kigamijwe mu gukora igenzura ryo ku rwego rw'itsinda.,True,False
11,train,Ingingo y


Law rows containing English text (dropped): 13957

Unique repaired Gazette records: 155771
Total repaired Gazette tokens: 2688772


In [9]:
law_candidate = take_until_token_target(
    law_df,
    target_tokens=CANDIDATE_TARGET,
    seed=42
)

law_candidate = (
    law_candidate
    .reset_index(drop=True)
)

law_candidate["id"] = [
    f"kin_law_{i:06d}"
    for i in range(
        1,
        len(law_candidate) + 1
    )
]

print(
    "Law candidate records:",
    len(law_candidate)
)

print(
    "Law candidate tokens:",
    int(
        law_candidate[
            "word_count"
        ].sum()
    )
)

law_candidate[COLUMNS].to_csv(
    "/kaggle/working/kin_law_candidates.csv",
    index=False,
    encoding="utf-8-sig"
)

print(
    "Saved: "
    "/kaggle/working/kin_law_candidates.csv"
)

Law candidate records: 17305
Law candidate tokens: 300006
Saved: /kaggle/working/kin_law_candidates.csv


# Part C — Telecoms: KINNEWS Technology only

## 5. Download KINNEWS

The official KINNEWS project provides raw and cleaned datasets. This notebook needs the **raw** version because it contains the human-readable category fields such as `en_label`.

The notebook first attempts to download the dataset from the **official Google Drive folder linked by the official KINNEWS repository**. If that fails, it falls back to the previously used public ZIP mirror.

The subsequent encoding-repair step is run regardless of download source so that mojibake cannot silently enter the final corpus.

In [10]:
KINNEWS_DIR = (
    "/kaggle/working/KINNEWS_DATA"
)

OFFICIAL_KINNEWS_FOLDER = (
    "https://drive.google.com/drive/folders/"
    "1zxn0hgrOLlUsK5V0c7l71eAj1t2jiyox"
    "?usp=sharing"
)

def find_raw_kinnews_csvs(root):
    train_matches = glob.glob(
        f"{root}/**/raw/train.csv",
        recursive=True
    )
    test_matches = glob.glob(
        f"{root}/**/raw/test.csv",
        recursive=True
    )
    return train_matches, test_matches


train_matches, test_matches = (
    find_raw_kinnews_csvs(
        KINNEWS_DIR
    )
)

if not train_matches or not test_matches:

    os.makedirs(
        KINNEWS_DIR,
        exist_ok=True
    )

    print(
        "Trying official KINNEWS "
        "Google Drive folder..."
    )

    try:
        import gdown

        gdown.download_folder(
            url=OFFICIAL_KINNEWS_FOLDER,
            output=KINNEWS_DIR,
            quiet=False,
            use_cookies=False,
        )

        # Extract any ZIP files that the
        # official folder contains.
        for zip_path in glob.glob(
            f"{KINNEWS_DIR}/**/*.zip",
            recursive=True
        ):
            try:
                with zipfile.ZipFile(
                    zip_path,
                    "r"
                ) as zf:
                    zf.extractall(
                        os.path.dirname(
                            zip_path
                        )
                    )
            except zipfile.BadZipFile:
                pass

    except Exception as exc:
        print(
            "Official download failed:",
            exc
        )


train_matches, test_matches = (
    find_raw_kinnews_csvs(
        KINNEWS_DIR
    )
)

# Fallback: previously used public mirror.
if not train_matches or not test_matches:

    print(
        "Falling back to public "
        "KINNEWS ZIP mirror..."
    )

    mirror_zip = (
        "/kaggle/working/"
        "KINNEWS_mirror.zip"
    )

    subprocess.run(
        [
            "wget",
            "-q",
            "-O",
            mirror_zip,
            (
                "https://github.com/"
                "saradhix/kinnews_kirnews/"
                "raw/master/KINNEWS.zip"
            ),
        ],
        check=True,
    )

    with zipfile.ZipFile(
        mirror_zip,
        "r"
    ) as zf:
        zf.extractall(
            KINNEWS_DIR
        )


train_matches, test_matches = (
    find_raw_kinnews_csvs(
        KINNEWS_DIR
    )
)

if not train_matches or not test_matches:
    raise FileNotFoundError(
        "KINNEWS raw/train.csv and "
        "raw/test.csv could not be found."
    )

print("Raw train:", train_matches[0])
print("Raw test :", test_matches[0])

Trying official KINNEWS Google Drive folder...


Retrieving folder contents


Retrieving folder 1-4YN6of9LFpHr0bYnjAz2GH-XsA_dulT .ipynb_checkpoints
Retrieving folder 1ijsLAHA6dSwM5tJdQgX9A2-UJ6JGS6Zl cleaned
Processing file 1wqyJnp8FCaGs8GA8XcsXiZQ-S-dBqEmQ test.csv
Processing file 1sw_VKfFS3HCm93s0Jo7p4ZOcgFoxbnbV train.csv
Retrieving folder 1rpHOP0aIpAEQuOe9yYISSrFblDFnUMEi raw
Retrieving folder 1DJQJBn1MJYW-cmHfU7woTFfRNDWkFLIV .ipynb_checkpoints
Processing file 1-0BIVpPXFAZ_3MPSN1t5Eep-QueU8vjb test.csv
Processing file 1---6X61jjAaf49G2bW8KFHD9b_Jquiux train.csv
Processing file 1Bxsw3vWCOZM8_sX7iLb79lnOYdOQvJsh classes.txt
Processing file 1zcCjFNx2ZNZ1wcFwhqLdsfytkUb_ItVB readme.txt


Retrieving folder contents completed
Building directory structure
Building directory structure completed
Downloading...
From: https://drive.google.com/uc?id=1wqyJnp8FCaGs8GA8XcsXiZQ-S-dBqEmQ
To: /kaggle/working/KINNEWS_DATA/cleaned/test.csv
100%|██████████| 8.17M/8.17M [00:00<00:00, 103MB/s]
Downloading...
From: https://drive.google.com/uc?id=1sw_VKfFS3HCm93s0Jo7p4ZOcgFoxbnbV
To: /kaggle/working/KINNEWS_DATA/cleaned/train.csv
100%|██████████| 32.6M/32.6M [00:00<00:00, 118MB/s] 
Downloading...
From: https://drive.google.com/uc?id=1-0BIVpPXFAZ_3MPSN1t5Eep-QueU8vjb
To: /kaggle/working/KINNEWS_DATA/raw/test.csv
100%|██████████| 11.9M/11.9M [00:00<00:00, 127MB/s]
Downloading...
From: https://drive.google.com/uc?id=1---6X61jjAaf49G2bW8KFHD9b_Jquiux
To: /kaggle/working/KINNEWS_DATA/raw/train.csv
100%|██████████| 38.0M/38.0M [00:00<00:00, 157MB/s]
Downloading...
From: https://drive.google.com/uc?id=1Bxsw3vWCOZM8_sX7iLb79lnOYdOQvJsh
To: /kaggle/working/KINNEWS_DATA/classes.txt
100%|██████████| 

Raw train: /kaggle/working/KINNEWS_DATA/raw/train.csv
Raw test : /kaggle/working/KINNEWS_DATA/raw/test.csv



Download completed


## 6. Load KINNEWS

The raw KINNEWS files are loaded as UTF-8. The same **global encoding-repair function already defined in Section 2** is then applied to both `title` and `content`.

Only `content` is later used as Telecom `raw_text`.

This shared repair approach ensures that **Law and Telecom are treated consistently** rather than using separate ad hoc fixes.

In [11]:
# Load the raw files as UTF-8.
train_df = pd.read_csv(
    train_matches[0],
    encoding="utf-8"
)

test_df = pd.read_csv(
    test_matches[0],
    encoding="utf-8"
)

train_df["source_split"] = "train"
test_df["source_split"] = "test"

print(
    "Train shape:",
    train_df.shape
)

print(
    "Test shape:",
    test_df.shape
)

print(
    "Columns:",
    train_df.columns.tolist()
)

Train shape: (17014, 7)
Test shape: (4254, 7)
Columns: ['label', 'kin_label', 'en_label', 'url', 'title', 'content', 'source_split']


### Global encoding helper

No second repair function is defined here. `repair_mojibake()` and `has_suspicious_mojibake()` were defined once in **Section 2** and are reused for both the Gazette and KINNEWS.

## 7. Apply the global encoding repair and audit KINNEWS

Both `title` and `content` are repaired, while only `content` becomes Telecom `raw_text`.

The audit reports:

1. how many KINNEWS rows changed;
2. how many still contain high-confidence mojibake markers;
3. before/after examples.

Do not treat residual suspicious rows as automatically wrong; inspect them. But clear sequences such as `caf√©` or `z‚Äô...` should not remain in accepted corpus text.

In [12]:
# Keep temporary originals for the audit.
train_df["_title_before"] = train_df["title"].copy()
train_df["_content_before"] = train_df["content"].copy()

test_df["_title_before"] = test_df["title"].copy()
test_df["_content_before"] = test_df["content"].copy()


# Extra targeted cleanup for the few remaining KINNEWS artifacts
def final_kinnews_cleanup(text):
    if not isinstance(text, str):
        return text

    # First use the general repair
    text = repair_mojibake(text)

    # Remaining artifacts observed in KINNEWS
    text = text.replace("Â ", " ")
    text = text.replace(" Â", " ")
    text = text.replace("Â", "")

    # Shared cleaning: NBSP, newlines, double spaces, quotes, emoji, Cyrillic
    return clean_text(text)


# Apply repair to both title and article content
for df in [train_df, test_df]:

    df["title"] = (
        df["title"]
        .apply(final_kinnews_cleanup)
    )

    df["content"] = (
        df["content"]
        .apply(final_kinnews_cleanup)
    )


# Combine train and test AFTER repair
kinnews = pd.concat(
    [train_df, test_df],
    ignore_index=True
)


# Track which rows changed
kinnews["encoding_changed"] = (
    (kinnews["title"] != kinnews["_title_before"])
    |
    (kinnews["content"] != kinnews["_content_before"])
)


print(
    "Total KINNEWS rows:",
    len(kinnews)
)

print(
    "Rows changed by encoding repair:",
    int(kinnews["encoding_changed"].sum())
)

Total KINNEWS rows: 21268
Rows changed by encoding repair: 21268


In [13]:
kinnews[
    "still_suspicious"
] = kinnews.apply(
    lambda row:
        has_suspicious_mojibake(
            row["content"]
        ),
    axis=1
)

remaining_suspicious = kinnews[
    kinnews["still_suspicious"]
].copy()

print(
    "KINNEWS rows still containing "
    "suspicious mojibake:",
    len(remaining_suspicious)
)

# Show before/after examples from rows
# that were actually repaired.
changed_sample = kinnews[
    kinnews["encoding_changed"]
][
    [
        "source_split",
        "_title_before",
        "title",
        "_content_before",
        "content",
    ]
].head(5)

display(changed_sample)

if len(remaining_suspicious) > 0:

    print(
        "\nResidual KINNEWS rows to inspect "
        "before final acceptance:"
    )

    display(
        remaining_suspicious[
            [
                "source_split",
                "title",
                "content",
            ]
        ].head(20)
    )
else:
    print(
        "\nNo high-confidence KINNEWS "
        "mojibake markers remain."
    )

KINNEWS rows still containing suspicious mojibake: 0


,source_split,_title_before,title,_content_before,content
0,train,Yihakanye uwo yabyaye animisha na nyina ibyo yagombaga guhabwa nk’abandi batishoboye,Yihakanye uwo yabyaye animisha na nyina ibyo yagombaga guhabwa nk'abandi batishoboye,"Icyatumye amwihakana ngo ni uko yanze kujya akomeza kuryamana nawe, nk’uko nyirubwite abivuga. Ati: ”Umugabo wanjye w’iserazerano aba Uganda, nkaba narabyaranye n’undi, uwo twabyaranye rero ni umuyobozi w’umudugudu wa kabuye. Nyuma aho tubyaraniye yashatse gukomeza kunsambanya ngo angire akarim...","Icyatumye amwihakana ngo ni uko yanze kujya akomeza kuryamana nawe, nk'uko nyirubwite abivuga. Ati: ""Umugabo wanjye w'iserazerano aba Uganda, nkaba narabyaranye n'undi, uwo twabyaranye rero ni umuyobozi w'umudugudu wa kabuye. Nyuma aho tubyaraniye yashatse gukomeza kunsambanya ngo angire akarima..."
1,train,Barasaba ko ba kavukire boroherezwa ntibakomeze kwimurwa mu mijyi,Barasaba ko ba kavukire boroherezwa ntibakomeze kwimurwa mu mijyi,"Iyi nama yateraniye mu cyumba cy’inama cya Bonni Concilli tariki 14/06/2012, yari igamije kungurana ibitekerezo ku myubakire ikwiye ituma amazu y’Abanyaturarwanda adasenyuka kubera ibiza. Nk’uko abari mu nama babigaragaje, ngo byakunze kugaragara ko uko imijyi igenda ikura hari abantu bagenda b...","Iyi nama yateraniye mu cyumba cy'inama cya Bonni Concilli tariki 14/06/2012, yari igamije kungurana ibitekerezo ku myubakire ikwiye ituma amazu y'Abanyaturarwanda adasenyuka kubera ibiza. Nk'uko abari mu nama babigaragaje, ngo byakunze kugaragara ko uko imijyi igenda ikura hari abantu bagenda bi..."
2,train,Abantu 9 bashakishwa cyane na ICTR na USA bashinjwa kugira uruhare muri Jenoside yakorewe abatutsi,Abantu 9 bashakishwa cyane na ICTR na USA bashinjwa kugira uruhare muri Jenoside yakorewe abatutsi,"Leta zunze Ubumwe z’Amerika kandi zanijeje uzatanga amakuru ko izina rye rizagirwa ibanga ndetse akazacungirwa umutekano w’aho azaba ari kuri iyi isi n’aho azaba ashaka kujya. Uza ku mwanya wa mbere muri abo barashakishwa, ni umunyemari Kabuga Felicien, uregwa kuba yaratanze inkunga y’ibikoresh...","Leta zunze Ubumwe z'Amerika kandi zanijeje uzatanga amakuru ko izina rye rizagirwa ibanga ndetse akazacungirwa umutekano w'aho azaba ari kuri iyi isi n'aho azaba ashaka kujya. Uza ku mwanya wa mbere muri abo barashakishwa, ni umunyemari Kabuga Felicien, uregwa kuba yaratanze inkunga y'ibikoresho..."
3,train,Yihakanye uwo yabyaye animisha na nyina ibyo yagombaga guhabwa nk’abandi batishoboye,Yihakanye uwo yabyaye animisha na nyina ibyo yagombaga guhabwa nk'abandi batishoboye,"Icyatumye amwihakana ngo ni uko yanze kujya akomeza kuryamana nawe, nk’uko nyirubwite abivuga. Ati: ”Umugabo wanjye w’iserazerano aba Uganda, nkaba narabyaranye n’undi, uwo twabyaranye rero ni umuyobozi w’umudugudu wa kabuye. Nyuma aho tubyaraniye yashatse gukomeza kunsambanya ngo angire akarim...","Icyatumye amwihakana ngo ni uko yanze kujya akomeza kuryamana nawe, nk'uko nyirubwite abivuga. Ati: ""Umugabo wanjye w'iserazerano aba Uganda, nkaba narabyaranye n'undi, uwo twabyaranye rero ni umuyobozi w'umudugudu wa kabuye. Nyuma aho tubyaraniye yashatse gukomeza kunsambanya ngo angire akarima..."
4,train,Barasaba ko ba kavukire boroherezwa ntibakomeze kwimurwa mu mijyi,Barasaba ko ba kavukire boroherezwa ntibakomeze kwimurwa mu mijyi,"Iyi nama yateraniye mu cyumba cy’inama cya Bonni Concilli tariki 14/06/2012, yari igamije kungurana ibitekerezo ku myubakire ikwiye ituma amazu y’Abanyaturarwanda adasenyuka kubera ibiza. Nk’uko abari mu nama babigaragaje, ngo byakunze kugaragara ko uko imijyi igenda ikura hari abantu bagenda b...","Iyi nama yateraniye mu cyumba cy'inama cya Bonni Concilli tariki 14/06/2012, yari igamije kungurana ibitekerezo ku myubakire ikwiye ituma amazu y'Abanyaturarwanda adasenyuka kubera ibiza. Nk'uko abari mu nama babigaragaje, ngo byakunze kugaragara ko uko imijyi igenda ikura hari abantu bagenda bi..."



No high-confidence KINNEWS mojibake markers remain.


In [14]:
for idx, row in remaining_suspicious.iterrows():
    print("=" * 100)
    print("INDEX:", idx)
    print("SPLIT:", row["source_split"])

    print("\nTITLE:")
    print(repr(row["title"]))

    print("\nCONTENT:")
    print(repr(row["content"]))

    print()

### Optional check for the Cyber café example

If that article exists in the downloaded files, this cell should show readable text such as `café`, `z’ubucuruzi`, and `« ... »`, rather than `caf√©`, `z‚Äô...`, or `¬´ ... ¬ª`.

In [16]:
cyber_example = kinnews[
    kinnews["content"]
    .fillna("")
    .str.contains(
        "Cyber café Ikaze",
        regex=False
    )
][
    [
        "title",
        "content",
        "source_split",
    ]
]

display(
    cyber_example.head(3)
)

,title,content,source_split
415,Muhanga: Gukoresha internet ntibikiri umwihariko w'Abanyakigali,"Cyber café Ikaze ni imwe mu nzu z'ubucuruzi bwa internet mu karere ka Muhanga. Ubuyobozi bwayo buvuga ko nubwo bizwi ko i Kigali ariho hari iterambere mu ikoranabuhanga ko n' i Muhanga n'aho abantu baryitabira cyane. Umukozi wa cyber café Ikaze agira ati: "" Abakiriya baritabira kugura serivisi y...",train
423,Muhanga: Gukoresha internet ntibikiri umwihariko w'Abanyakigali,"Cyber café Ikaze ni imwe mu nzu z'ubucuruzi bwa internet mu karere ka Muhanga. Ubuyobozi bwayo buvuga ko nubwo bizwi ko i Kigali ariho hari iterambere mu ikoranabuhanga ko n' i Muhanga n'aho abantu baryitabira cyane. Umukozi wa cyber café Ikaze agira ati: "" Abakiriya baritabira kugura serivisi y...",train
431,Muhanga: Gukoresha internet ntibikiri umwihariko w'Abanyakigali,"Cyber café Ikaze ni imwe mu nzu z'ubucuruzi bwa internet mu karere ka Muhanga. Ubuyobozi bwayo buvuga ko nubwo bizwi ko i Kigali ariho hari iterambere mu ikoranabuhanga ko n' i Muhanga n'aho abantu baryitabira cyane. Umukozi wa cyber café Ikaze agira ati: "" Abakiriya baritabira kugura serivisi y...",train


## 8. Keep only the KINNEWS Technology category

The readable category label comes from the raw dataset's `en_label` field.

The article **body (`content`)**, not the title, becomes `raw_text`.

Exact duplicate article bodies are removed **after encoding repair**. This avoids counting repeated text as additional corpus material.

In [17]:
technology_all = kinnews[
    kinnews["en_label"]
    .astype(str)
    .str.strip()
    .str.lower()
    .eq("technology")
].copy()


technology_all = technology_all[
    technology_all["content"]
    .apply(
        lambda x:
        isinstance(x, str)
        and bool(x.strip())
    )
].copy()


print(
    "Technology rows before "
    "exact deduplication:",
    len(technology_all)
)


technology_df = (
    technology_all
    .drop_duplicates(
        subset=["content"]
    )
    .reset_index(drop=True)
)


technology_df["word_count"] = (
    technology_df["content"]
    .apply(count_words)
)


print(
    "Unique Technology records:",
    len(technology_df)
)

print(
    "Duplicate Technology rows removed:",
    len(technology_all)
    - len(technology_df)
)

print(
    "Corrected Technology tokens:",
    int(
        technology_df[
            "word_count"
        ].sum()
    )
)

print(
    "Average tokens/article:",
    round(
        technology_df[
            "word_count"
        ].mean(),
        1
    )
)

print(
    "Remaining to 200,000:",
    max(
        0,
        FINAL_TARGET
        - int(
            technology_df[
                "word_count"
            ].sum()
        )
    )
)

Technology rows before exact deduplication: 438
Unique Technology records: 253
Duplicate Technology rows removed: 185
Corrected Technology tokens: 90520
Average tokens/article: 357.8
Remaining to 200,000: 109480


> **Important:** Earlier exploratory counts such as `90,522` Telecom tokens were calculated before the full substring-level encoding repair. Use the **corrected token count printed by the cell above** as the authoritative Telecom count for this notebook.

In [18]:
kinnews_telecom = (
    technology_df
    .copy()
    .reset_index(drop=True)
)


kinnews_telecom["id"] = [
    f"kin_telecoms_kinnews_{i:06d}"
    for i in range(
        1,
        len(kinnews_telecom) + 1
    )
]


kinnews_telecom["language"] = (
    "kin"
)

kinnews_telecom["domain"] = (
    "telecoms"
)


kinnews_telecom["raw_text"] = (
    kinnews_telecom["content"]
    .fillna("")
    .astype(str)
    .str.strip()
)


kinnews_telecom[
    "source_dataset"
] = "KINNEWS"


kinnews_telecom[
    "license"
] = "MIT"


kinnews_telecom[
    "review_status"
] = "unreviewed"


kinnews_telecom[
    "notes"
] = (
    "Originally labelled Technology "
    "in KINNEWS; retained under the "
    "project's Telecom/Technology scope. "
    "Source-level mojibake repair and clean_text() "
    "applied before deduplication."
)


kinnews_telecom = (
    kinnews_telecom[
        COLUMNS + ["word_count"]
    ]
)


print(
    "Telecom records:",
    len(kinnews_telecom)
)

print(
    "Telecom tokens:",
    int(
        kinnews_telecom[
            "word_count"
        ].sum()
    )
)


kinnews_telecom[
    COLUMNS
].to_csv(
    "/kaggle/working/"
    "kinnews_technology_telecom_candidates.csv",
    index=False,
    encoding="utf-8-sig"
)


print(
    "Saved: /kaggle/working/"
    "kinnews_technology_"
    "telecom_candidates.csv"
)

Telecom records: 253
Telecom tokens: 90520
Saved: /kaggle/working/kinnews_technology_telecom_candidates.csv


# Part D — Combine all five domains

## 9. Create the combined candidate corpus

The combined corpus contains:

- Health — Afrivoice
- Finance — Afrivoice
- Agriculture — Afrivoice
- Law — Rwanda Gazette
- Telecoms — **unique, encoding-repaired KINNEWS Technology article content only**

In [19]:
all_candidates = pd.concat(
    [
        candidate_hfa[COLUMNS],
        law_candidate[COLUMNS],
        kinnews_telecom[COLUMNS],
    ],
    ignore_index=True
)


# Remove exact duplicates within a domain.
all_candidates = (
    all_candidates
    .drop_duplicates(
        subset=[
            "domain",
            "raw_text"
        ],
        keep="first"
    )
    .reset_index(drop=True)
)


all_candidates["word_count"] = (
    all_candidates["raw_text"]
    .apply(count_words)
)


domain_progress = (
    all_candidates
    .groupby("domain")
    .agg(
        records=("raw_text", "size"),
        tokens=("word_count", "sum"),
    )
)


domain_progress["target"] = (
    FINAL_TARGET
)


domain_progress["remaining"] = (
    domain_progress["target"]
    - domain_progress["tokens"]
).clip(lower=0)


domain_progress["coverage_%"] = (
    domain_progress["tokens"]
    / domain_progress["target"]
    * 100
).round(1)


display(domain_progress)

,records,tokens,target,remaining,coverage_%
domain,,,,,
agriculture,11520,300022,200000,0,150.0
finance,10902,300015,200000,0,150.0
health,10731,300015,200000,0,150.0
law,17305,300006,200000,0,150.0
telecoms,253,90520,200000,109480,45.3


# Part E — Quality audit

## 10. Automatic quality indicators

These checks help identify records that may need review. They do not automatically determine linguistic correctness.

In [20]:
def has_punctuation(text):
    return bool(
        re.search(
            r"[.!?,;:]",
            str(text)
        )
    )


def starts_with_capital(text):
    text = str(text).strip()

    for char in text:
        if char.isalpha():
            return char.isupper()

    return False


audit_df = (
    all_candidates.copy()
)


audit_df[
    "has_punctuation"
] = (
    audit_df["raw_text"]
    .apply(has_punctuation)
)


audit_df[
    "starts_capital"
] = (
    audit_df["raw_text"]
    .apply(starts_with_capital)
)


audit_df[
    "multiple_spaces"
] = (
    audit_df["raw_text"]
    .str.contains(
        r"\s{2,}",
        regex=True
    )
)


audit_df[
    "space_before_punctuation"
] = (
    audit_df["raw_text"]
    .str.contains(
        r"\s+[,.!?;:]",
        regex=True
    )
)


audit_df[
    "very_short"
] = (
    audit_df["word_count"] < 5
)


quality_summary = (
    audit_df
    .groupby("domain")
    .agg(
        records=("raw_text", "size"),
        tokens=("word_count", "sum"),
        punctuation_coverage=(
            "has_punctuation",
            "mean"
        ),
        starts_capital=(
            "starts_capital",
            "mean"
        ),
        multiple_spaces=(
            "multiple_spaces",
            "mean"
        ),
        space_before_punctuation=(
            "space_before_punctuation",
            "mean"
        ),
        very_short=(
            "very_short",
            "mean"
        ),
    )
)


percentage_columns = [
    "punctuation_coverage",
    "starts_capital",
    "multiple_spaces",
    "space_before_punctuation",
    "very_short",
]


for column in percentage_columns:

    quality_summary[column] = (
        quality_summary[column]
        * 100
    ).round(2)


display(quality_summary)

,records,tokens,punctuation_coverage,starts_capital,multiple_spaces,space_before_punctuation,very_short
domain,,,,,,,
agriculture,11520,300022,99.93,99.49,0.0,4.12,0.02
finance,10902,300015,99.78,99.28,0.0,3.60,0.01
health,10731,300015,99.80,99.52,0.0,1.20,0.00
law,17305,300006,95.51,68.88,0.0,0.00,7.17
telecoms,253,90520,100.00,99.60,0.0,0.00,0.00


# Part F — Manual review

## 11. Create a balanced 100-record review sample

The sample contains **20 records from each of the five domains = 100 total**.

Review fields:

- `Is_Kin`: `YES`, `NO`, or `CODE-SWITCHING`
- `domain_ok`: `YES` / `NO`
- `punctuation_ok`: `YES` / `NO`
- `quality_ok`: `YES` / `NO`
- `keep`: `YES` / `NO`
- `review_status`: set to `YES` only after that row has been reviewed
- `review_notes`: short explanation where useful

For Law, isolated list fragments can be flagged for structural cleaning or rejected if they are not meaningful independently.

In [21]:
review_frames = []

SAMPLES_PER_DOMAIN = 20


for domain in DOMAINS:

    domain_df = all_candidates[
        all_candidates[
            "domain"
        ] == domain
    ].copy()

    if len(domain_df) == 0:
        continue

    sample = domain_df.sample(
        n=min(
            SAMPLES_PER_DOMAIN,
            len(domain_df)
        ),
        random_state=42
    ).copy()

    sample["Is_Kin"] = ""
    sample["domain_ok"] = ""
    sample["punctuation_ok"] = ""
    sample["quality_ok"] = ""
    sample["keep"] = ""
    sample["review_status"] = "NO"
    sample["review_notes"] = ""

    review_frames.append(sample)


manual_review_sample = pd.concat(
    review_frames,
    ignore_index=True
)


print(
    "Total records for manual review:",
    len(manual_review_sample)
)

print("\nRecords by domain:")

print(
    manual_review_sample[
        "domain"
    ].value_counts()
)


display(
    manual_review_sample[
        [
            "id",
            "domain",
            "raw_text",
            "source_dataset",
            "license",
            "review_status",
            "Is_Kin",
            "domain_ok",
            "punctuation_ok",
            "quality_ok",
            "keep",
            "review_notes",
        ]
    ]
)

Total records for manual review: 100

Records by domain:
domain
health         20
finance        20
agriculture    20
telecoms       20
law            20
Name: count, dtype: int64


,id,domain,raw_text,source_dataset,license,review_status,Is_Kin,domain_ok,punctuation_ok,quality_ok,keep,review_notes
0,kin_health_000593,health,Imbere y'ameza aramutse y'impapuro hicaye umugore uri kumwe n'umugabo ukuze ufite terefone mu ntoki wambaye ishati y'amaboko magufi irimo umurongo n'amataratara . Inyuma yabo hari intebe nyinshi inyuma y'izo ntebe hari ibirahure biri gutuma urumuri rwinjira ahantu bicaye.,DigitalUmuganda/Afrivoice_Kinyarwanda,CC-BY-4.0,NO,,,,,,
1,kin_health_000609,health,"Imiti myinshi muri rusange, usanga iba yaragaragajwe cyangwa yaravumbuwe n'abantu b'abahanga kandi bakagaragariza abantu cyangwa leta n'abaturage, mu rwego rwo kuyishyira ku isoko kugira ngo ibashe gufasha mu kuvura abantu ndetse no kubahiriza ubuzima bwiza.",DigitalUmuganda/Afrivoice_Kinyarwanda,CC-BY-4.0,NO,,,,,,
2,kin_health_000425,health,"Abana bari ku ivomero harimo uwambaye ikabutura y'umukara ndetse n'agapira k'amaboko magufi gafite ibara ry'umuhondo, hirya ye hari umugore wambaye ikanzu ndetse mu mutwe atezemo agatambaro gafite ibara ry'ubururu.",DigitalUmuganda/Afrivoice_Kinyarwanda,CC-BY-4.0,NO,,,,,,
3,kin_health_004182,health,"Minisiteri y'ubuzima mu Rwanda ikomeje gushishikariza abaturage bose kwitabira ibikorwa bya siporo, kuko byagaragaye ko siporo ari ingenzi mu buzima kandi ifasha umuntu uyikora no kumurinda uburwayi butandukanye.",DigitalUmuganda/Afrivoice_Kinyarwanda,CC-BY-4.0,NO,,,,,,
4,kin_health_005637,health,"Umuganga wunamye, arimo arumva umurwayi uko ameze kugira ngo abone indwara amuvura, yambaye itaburiya y'ubururu ifite akarongo k'iroza ku kuboko.",DigitalUmuganda/Afrivoice_Kinyarwanda,CC-BY-4.0,NO,,,,,,
...,...,...,...,...,...,...,...,...,...,...,...,...
95,kin_law_007727,law,5° gukora akazi kose kihariye yashingwa n'umuhuzabikorwa wa dasso kajyanye n'inshingano za dasso.,DigitalUmuganda/NMT_Rwandan-Gazette_parallel_data_en_kin,CC-BY-4.0,NO,,,,,,
96,kin_law_005772,law,"Amabwiriza ya perezida w'urukiko rw'ikirenga agenga imiburanishirize y'imanza z'imbonezamubano, iz'ubucuruzi, iz'umurimo n'iz'ubutegetsi...................................37",DigitalUmuganda/NMT_Rwandan-Gazette_parallel_data_en_kin,CC-BY-4.0,NO,,,,,,
97,kin_law_016337,law,Itegeko n° 25/2015 ryo ku wa 23/05/2015 ryemerera kwemeza burundu amasezerano ajyanye no gutwara abantu n'ibintu mu kirere yashyiriweho umukono i jeddah muri arabie saoudite ku wa 12 ukuboza 2012 hagati ya guverinoma y'u rwanda na guverinoma ya swaziland,DigitalUmuganda/NMT_Rwandan-Gazette_parallel_data_en_kin,CC-BY-4.0,NO,,,,,,
98,kin_law_016777,law,7° kwemeza impano n'indagano umurenge ushobora guhabwa;,DigitalUmuganda/NMT_Rwandan-Gazette_parallel_data_en_kin,CC-BY-4.0,NO,,,,,,


## Character verification (Law + Telecom)

Run before export. Only real letters and a few symbols (`é è ï ç ° £ ¼`) should remain, and both space counts should be 0.


In [22]:
for d in ["law", "telecoms"]:
    s = all_candidates.loc[all_candidates.domain == d, "raw_text"]
    chars = sorted({c for t in s for c in t if ord(c) > 127})
    print(d, "non-ASCII characters left:", chars)
    print(
        d,
        "| double-space rows:", int(s.str.contains(r"\s{2,}").sum()),
        "| space-before-punct rows:", int(s.str.contains(r"\s[,.!?;:]").sum()),
        "| newline rows:", int(s.str.contains(r"[\r\n\t]").sum()),
    )


law non-ASCII characters left: ['¤', '¥', '§', '¨', '®', '°', '´', '¼', '½', '¾', '×', 'à', 'ä', 'ç', 'è', 'é', 'ê', 'ë', 'í', 'ï', 'ô', 'ù', 'ú', 'œ', 'ŕ', 'ʼ', '˂', '˃', '˚', 'ˮ', '΄', 'ᵒ', '―', '•', '′', '″', '‹', '›', '⁰', '№', '⅔', '−']
law | double-space rows: 0 | space-before-punct rows: 0 | newline rows: 0
telecoms non-ASCII characters left: ['£', '°', '¼', '¾', 'à', 'â', 'ç', 'è', 'é', 'í', 'ï', 'ô', 'ü']
telecoms | double-space rows: 0 | space-before-punct rows: 0 | newline rows: 0


# Part G — Export

## 12. Save candidate corpus, progress, quality, review, and encoding-audit files

The JSONL contains only the shared Practicum schema fields.

Two source-level encoding audit files are saved:

- `law_encoding_audit.csv`
- `kinnews_encoding_audit.csv`

These make the encoding repair traceable rather than silently changing source text.

In [23]:
OUTPUT_DIR = Path(
    "/kaggle/working"
)

OUTPUT_DIR.mkdir(
    parents=True,
    exist_ok=True
)

combined_export = (
    all_candidates[COLUMNS]
    .copy()
)

csv_path = (
    OUTPUT_DIR
    / "kinyarwanda_multidomain_candidates.csv"
)

jsonl_path = (
    OUTPUT_DIR
    / "kinyarwanda_multidomain_candidates.jsonl"
)

progress_path = (
    OUTPUT_DIR
    / "kinyarwanda_domain_progress.csv"
)

quality_path = (
    OUTPUT_DIR
    / "kinyarwanda_quality_summary.csv"
)

review_path = (
    OUTPUT_DIR
    / "kinyarwanda_manual_review_100.csv"
)

law_encoding_audit_path = (
    OUTPUT_DIR
    / "law_encoding_audit.csv"
)

kinnews_encoding_audit_path = (
    OUTPUT_DIR
    / "kinnews_encoding_audit.csv"
)

combined_export.to_csv(
    csv_path,
    index=False,
    encoding="utf-8-sig"
)

combined_export.to_json(
    jsonl_path,
    orient="records",
    lines=True,
    force_ascii=False
)

domain_progress.to_csv(
    progress_path,
    encoding="utf-8-sig"
)

quality_summary.to_csv(
    quality_path,
    encoding="utf-8-sig"
)

manual_review_sample.to_csv(
    review_path,
    index=False,
    encoding="utf-8-sig"
)

# Law audit was created before Law deduplication.
law_encoding_audit.to_csv(
    law_encoding_audit_path,
    index=False,
    encoding="utf-8-sig"
)

kinnews_encoding_audit = kinnews[
    kinnews["encoding_changed"]
    | kinnews["still_suspicious"]
][
    [
        "source_split",
        "_title_before",
        "title",
        "_content_before",
        "content",
        "encoding_changed",
        "still_suspicious",
    ]
].copy()

kinnews_encoding_audit.to_csv(
    kinnews_encoding_audit_path,
    index=False,
    encoding="utf-8-sig"
)

print("Saved:")

for path in [
    csv_path,
    jsonl_path,
    progress_path,
    quality_path,
    review_path,
    law_encoding_audit_path,
    kinnews_encoding_audit_path,
]:
    print("-", path)

Saved:
- /kaggle/working/kinyarwanda_multidomain_candidates.csv
- /kaggle/working/kinyarwanda_multidomain_candidates.jsonl
- /kaggle/working/kinyarwanda_domain_progress.csv
- /kaggle/working/kinyarwanda_quality_summary.csv
- /kaggle/working/kinyarwanda_manual_review_100.csv
- /kaggle/working/law_encoding_audit.csv
- /kaggle/working/kinnews_encoding_audit.csv


In [24]:
# =========================================================
# MANUAL REVIEW SAMPLE — LAW + TELECOM ONLY
# 20 Law + 20 Telecom = 40 records
# =========================================================

review_40 = []

for domain in ["law", "telecoms"]:

    domain_df = all_candidates[
        all_candidates["domain"] == domain
    ].copy()

    sample = domain_df.sample(
        n=min(20, len(domain_df)),
        random_state=42
    ).copy()

    sample["Is_Kin"] = ""
    sample["domain_ok"] = ""
    sample["punctuation_ok"] = ""
    sample["quality_ok"] = ""
    sample["keep"] = ""
    sample["review_status"] = "NO"
    sample["review_notes"] = ""

    review_40.append(sample)


law_telecom_review = pd.concat(
    review_40,
    ignore_index=True
)

print("Total review records:", len(law_telecom_review))

print("\nBy domain:")
print(
    law_telecom_review["domain"]
    .value_counts()
)

display(
    law_telecom_review[
        [
            "id",
            "language",
            "domain",
            "raw_text",
            "source_dataset",
            "license",
            "review_status",
            "Is_Kin",
            "domain_ok",
            "punctuation_ok",
            "quality_ok",
            "keep",
            "review_notes",
        ]
    ]
)

Total review records: 40

By domain:
domain
law         20
telecoms    20
Name: count, dtype: int64


,id,language,domain,raw_text,source_dataset,license,review_status,Is_Kin,domain_ok,punctuation_ok,quality_ok,keep,review_notes
0,kin_law_008969,kin,law,(d) umuyobozi wa polisi mu karere;,DigitalUmuganda/NMT_Rwandan-Gazette_parallel_data_en_kin,CC-BY-4.0,NO,,,,,,
1,kin_law_005937,kin,law,"Icyakora, iyo perezida wa repubulika yakatiwe igihano n'inkiko kubera kugambanira igihugu cyangwa kwica bikomeye kandi nkana itegeko nshinga, ntashobora guhabwa ibigenerwa abacyuye igihe.",DigitalUmuganda/NMT_Rwandan-Gazette_parallel_data_en_kin,CC-BY-4.0,NO,,,,,,
2,kin_law_006149,kin,law,Inama y'ubuyobozi y'ikigo igizwe n'abantu barindwi (7).,DigitalUmuganda/NMT_Rwandan-Gazette_parallel_data_en_kin,CC-BY-4.0,NO,,,,,,
3,kin_law_008301,kin,law,Ingingo ya 428: gutwika imyanda no guhumanya ikirere,DigitalUmuganda/NMT_Rwandan-Gazette_parallel_data_en_kin,CC-BY-4.0,NO,,,,,,
4,kin_law_011199,kin,law,Agizwe umuyobozi wa politike n'igenamigambi muri minisiteri y' urubyiruko.,DigitalUmuganda/NMT_Rwandan-Gazette_parallel_data_en_kin,CC-BY-4.0,NO,,,,,,
5,kin_law_015358,kin,law,"4° urutonde rwuzuye rw'abagenewe amakuru bwite cyangwa abazayahabwa, harimo abatuye mu bindi bihugu;",DigitalUmuganda/NMT_Rwandan-Gazette_parallel_data_en_kin,CC-BY-4.0,NO,,,,,,
6,kin_law_016038,kin,law,Ingingo ya 68: igisobanuro cy'imbabazi zitangwa n'itegeko,DigitalUmuganda/NMT_Rwandan-Gazette_parallel_data_en_kin,CC-BY-4.0,NO,,,,,,
7,kin_law_005032,kin,law,Ingingo ya 56: igihe cyo gutanga ikiruhuko cy'ingoboka,DigitalUmuganda/NMT_Rwandan-Gazette_parallel_data_en_kin,CC-BY-4.0,NO,,,,,,
8,kin_law_002572,kin,law,"Ingingo ya 21: ifatira ry'agateganyo n'ubunyagwe bw'ahantu hakorerwa icuruzwa ry'abantu, ubw'ibikomoka ku cyaha n'ibintu byakoreshejwe mu cyaha",DigitalUmuganda/NMT_Rwandan-Gazette_parallel_data_en_kin,CC-BY-4.0,NO,,,,,,
9,kin_law_015865,kin,law,"Uwashinzwe gucunga ingwate ni we ushyikiriza uwatanze ingwate na nyir'umutungo iyo anyuranye n'uwatanze ingwate kopi y'icyemezo cyo gucunga ingwate, mu gihe kitarenze iminsi itanu (5) uhereye ku munsi uwashinzwe gucunga ingwate yamenyeyeho ishyirwaho rye.",DigitalUmuganda/NMT_Rwandan-Gazette_parallel_data_en_kin,CC-BY-4.0,NO,,,,,,


In [25]:
law_telecom_review[
    [
        "id",
        "language",
        "domain",
        "raw_text",
        "source_dataset",
        "license",
        "review_status",
        "Is_Kin",
        "domain_ok",
        "punctuation_ok",
        "quality_ok",
        "keep",
        "review_notes",
    ]
].to_csv(
    "/kaggle/working/law_telecom_manual_review_40.csv",
    index=False,
    encoding="utf-8-sig"
)

print(
    "Saved: /kaggle/working/law_telecom_manual_review_40.csv"
)

Saved: /kaggle/working/law_telecom_manual_review_40.csv


## 13. Verify JSONL and package all outputs

The ZIP is the easiest file to download locally from Kaggle.

In [26]:
print("First 3 JSONL records:\n")

with open(
    "/kaggle/working/"
    "kinyarwanda_multidomain_candidates.jsonl",
    "r",
    encoding="utf-8"
) as file:

    for _ in range(3):
        line = file.readline().strip()

        if line:
            print(line)

zip_path = (
    "/kaggle/working/"
    "kinyarwanda_outputs.zip"
)

files_to_zip = [
    str(csv_path),
    str(jsonl_path),
    str(progress_path),
    str(quality_path),
    str(review_path),
    str(law_encoding_audit_path),
    str(kinnews_encoding_audit_path),
    (
        "/kaggle/working/"
        "kin_hfa_candidates.csv"
    ),
    (
        "/kaggle/working/"
        "kin_law_candidates.csv"
    ),
    (
        "/kaggle/working/"
        "kinnews_technology_"
        "telecom_candidates.csv"
    ),
]

with zipfile.ZipFile(
    zip_path,
    "w",
    compression=zipfile.ZIP_DEFLATED
) as zf:

    for file_path in files_to_zip:

        if os.path.exists(file_path):

            zf.write(
                file_path,
                arcname=os.path.basename(
                    file_path
                )
            )

print(
    "\nZIP created:",
    zip_path
)

from IPython.display import (
    FileLink,
    display
)

display(
    FileLink(zip_path)
)

First 3 JSONL records:

{"id":"kin_health_000001","language":"kin","domain":"health","raw_text":"Ibinini by'umuhondo, umweru w'umutuku ndetse n'andi mabara byakoze ishusho y'umuntu uri kwirukanka, iruhande hari n'inshinge zizwi nka serenge.","source_dataset":"DigitalUmuganda\/Afrivoice_Kinyarwanda","source_split":"train","license":"CC-BY-4.0","review_status":"unreviewed","notes":"Loaded through Professor\/kinyarwanda-speech-data manifest; verify human-reference transcript provenance before final acceptance."}
{"id":"kin_health_000002","language":"kin","domain":"health","raw_text":"Icyumba cyo kwa muganga kirimo ibikoresho bigezweho, byifashisha mu kuvura umurwayi kugira ngo abashe kubona ubuvuzi bwizewe kandi bugezweho mu gihe cyihuse kitarambiranye kandi gitoya, harimo imashini zifashishwa mu gusuzuma umurwayi imbere mu mubiri, aribyo bita kumunyuza mu cyuma.","source_dataset":"DigitalUmuganda\/Afrivoice_Kinyarwanda","source_split":"train","license":"CC-BY-4.0","review_status":"unrevi

/kaggle/working/kinyarwanda_outputs.zip

# Current checkpoint

After running this notebook, use the **`domain_progress` table produced by the notebook** as the authoritative count.

Both **Law** and **KINNEWS Telecom/Technology** are now repaired with the same source-level encoding function **before deduplication and token counting**.

### Before accepting the corpus

Check these two outputs carefully:

```text
Law rows still containing suspicious mojibake: ...
KINNEWS rows still containing suspicious mojibake: ...
```

A non-zero number does not automatically mean every row is wrong, because the audit intentionally uses conservative markers. Inspect the displayed residual examples for genuine corruption.

### Remaining work

1. Inspect any residual encoding-audit rows.
2. Complete the 100-record manual review.
3. Apply agreed deterministic linguistic/structural cleaning rules.
4. Recalculate clean accepted token totals.
5. Add more Telecom/Technology material later if the unique corrected KINNEWS corpus remains below 200,000 tokens.
6. Verify the human-reference provenance of the speech-derived Afrivoice transcripts before final acceptance.

### Note on encoding

If `y‚Äôurugaga`, `caf√©` or `10¬∞` appear when you open a CSV, the file is being read as MacRoman by the spreadsheet app. All CSV exports here use `utf-8-sig`; in Excel you can also use **Data → From Text/CSV → File Origin: UTF-8**.
